# Build an autonomous bug investigator with Claude Managed Agents

## Why build this?

Most bug triage is a manual loop: read the report, find the right file, grep for the error, open a database client, check the audit trail, count affected rows, check if it's happened before, write it all up, file the ticket. That first-pass investigation eats 60–120 minutes per incident — and it's the *same* sequence of steps every time.

[Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview) lets you turn that sequence into an autonomous agent that runs in a cloud sandbox. You define a **Skill** (the investigation runbook), create an **Agent** (model + tools + skill), spin up an **Environment** (the sandbox), and drive work through **Sessions** and **Events**.

Here’s the full pipeline:

```text
        ┌────────────────────────────┐
        │    Bug report (webhook)    │
        └──────────────┬─────────────┘
                       │
        ┌──────────────▼─────────────┐
        │   Create session + mount   │
        │   codebase into sandbox    │
        └──────────────┬─────────────┘
                       │
        ┌──────────────▼─────────────┐
        │   Agent investigates       │
        │   grep, read, bash, edit   │
        └──────────────┬─────────────┘
                       │
                       ├──► query_database ·········· auto
                       ├──► search_past_incidents ··· auto
                       ├──► open_pull_request ······· auto
                       │
        ┌──────────────▼─────────────┐
        │   create_triage_ticket     │
        │       ■ HUMAN GATE ■       │
        └──────────────┬─────────────┘
                       │ approved
        ┌──────────────▼─────────────┐
        │  Generate remediation.sql  │
        │  (fix 12 corrupted orders) │
        └────────────────────────────┘
```

This cookbook turns that sequence into an agent. Instead of classifying bugs from text alone, the agent **investigates**: it reads source code in a sandbox, runs read-only SQL against your database, cross-references past incidents, **proposes a code fix**, **opens a pull request**, and produces a structured triage report with root cause, blast radius, and severity classification. A human reviews the findings and the PR before anything gets filed or merged.

The key design choices:

| Decision | Why |
|----------|-----|
| **Skill for the protocol** | The investigation runbook (severity matrix, query templates, report format) lives in a [Skill](https://platform.claude.com/docs/en/managed-agents/skills) — portable between Managed Agents, the Agent SDK, and Claude Code. |
| **Sandbox for code + fix** | The agent reads, greps, and **edits** the codebase using the built-in `agent_toolset` inside a cloud sandbox. The fix is produced as a unified diff — no direct access to production file systems. |
| **Custom tools for data + actions** | Database queries, issue-tracker search, and PR creation are custom tools your backend executes. You control the connection string, enforce read-only access, and decide what gets returned. |
| **Human gate for tickets** | `create_triage_ticket` pauses the session — the agent can’t file a ticket (or get the PR merged) without a human saying yes. |

Everything below runs with only `ANTHROPIC_API_KEY`. The database and issue tracker are mocked with local fixtures; the closing section shows where each mock plugs into a real service.

### How this relates to other Managed Agents cookbooks

| Cookbook | Focus | How it differs from this one |
|---------|-------|------------------------------|
| [SRE incident responder](sre_incident_responder.ipynb) | Infrastructure alert → read logs → open PR → approve → merge | Fixes infra config; no database access, no issue-tracker search. |
| [Human-in-the-loop gate](CMA_gate_human_in_the_loop.ipynb) | Teaches the `requires_action` pattern with expense receipts | Pedagogical; no Skill, no SQL, no investigation workflow. |
| [Data analyst agent](data_analyst_agent.ipynb) | CSV → HTML report with charts | Analytics output; no custom tools, no triage. |
| **This cookbook** | Application bug → **grep code + read-only SQL + past incidents → code fix + PR** → triage ticket | Full investigate-and-fix loop: sandbox code reading, external data tools, proposed fix with diff, and PR — then human approval. |

### What you'll build

1. A **Skill** encoding your team's investigation protocol, severity rules, and SQL query templates.
2. A **Managed Agent** that combines sandbox code-reading/editing tools with four custom tools (`query_database`, `search_past_incidents`, `open_pull_request`, `create_triage_ticket`).
3. An **event loop** that auto-answers data tools and the PR tool, and pauses at ticket creation for human approval.
4. A working demo where a planted discount-calculation bug is found, quantified, cross-referenced with a prior incident, **fixed in the sandbox**, submitted as a PR, reported, and a **data remediation script** generated to fix affected records — all autonomously.

### Prerequisites

You need **Python 3.11+** and an `ANTHROPIC_API_KEY`. Set the key in your environment (or a `.env` file), then install dependencies:

In [ ]:
%pip install -q "anthropic>=0.91.0" python-dotenv

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv
from utilities import wait_for_idle_status

load_dotenv()
client = Anthropic()
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")

# Fixtures live alongside the notebook under example_data/.
# They provide the bug report, mock DB rows, past incidents,
# and the Python source files that get mounted into the sandbox.
FIXTURE = Path("example_data/bug_investigator").resolve()
if str(FIXTURE) not in sys.path:
    sys.path.insert(0, str(FIXTURE))

from fixtures.database.fixtures import BUG_REPORT, PAST_INCIDENTS, execute_mock_query

## The scenario

A support agent reports: *"Checkout crashes with negative total when applying discount code BULK25."* Four customers have hit it in the last hour.

Behind the scenes, our fixture codebase has a planted bug: the `tiered_percent` discount type applies a per-item percentage without capping at `MAX_DISCOUNT_PERCENT`. For orders with many cheap items the discount exceeds the subtotal, producing a negative total that the payment gateway rejects.

The agent needs to:
1. **Grep the code** to find the discount calculation path and any TODOs.
2. **Query the database** to see the order, its items, the discount config, and the audit trail.
3. **Count the blast radius** — how many other orders are affected.
4. **Search past incidents** for anything similar (it will find SHOP-4891).
5. **Classify severity** using the Skill’s matrix.
6. **Edit the code** to fix the root cause and produce a unified diff.
7. **Open a pull request** with the fix via `open_pull_request`.
8. **Produce a triage report** linking the PR, and **pause for human approval** before filing the ticket.
9. **Generate a data remediation script** to fix the 12 orders already corrupted by the bug.

## 1. Upload the investigation skill

A Skill bundles team knowledge — investigation procedures, severity matrices, SQL templates, report structure — into a small file tree the platform mounts into the agent's context. The agent sees its one-line description immediately and reads the full body when it's relevant.

The skill here (`example_data/bug_investigator/skill/SKILL.md`) encodes nine investigation steps (including code fix, PR, and data remediation) and a severity matrix. The same file works if you use it locally in Claude Code or the Agent SDK, because it's just a markdown document with a clear protocol.

> **Note:** These are *Managed Agents Skills* — markdown runbooks uploaded via `beta.skills.create` and mounted into the agent's context. They are different from the office/finance Skills covered in the [`skills/` notebooks](../skills/notebooks/01_skills_introduction.ipynb).

In [1]:
SKILL_PATH = FIXTURE / "skill" / "SKILL.md"
SKILL_CONTENT = SKILL_PATH.read_text(encoding="utf-8")

skill = client.beta.skills.create(
    display_title="bug-investigator-runbook",
    files=[
        ("bug-investigator-runbook/SKILL.md", SKILL_CONTENT.encode(), "text/markdown"),
    ],
)
print(f"skill: {skill.id} (version {skill.latest_version})")

skill: skill_01JC8K9XVQE (version sv_01JC8K9Y2MN)


## 2. Create the agent with four custom tools

The agent needs to work across two boundaries: **code** (sandbox) and **data** (your backend). The tool list reflects that split:

- **`agent_toolset_20260401`** gives the agent `read`, `grep`, `bash`, and `edit` inside the sandbox. It will use these to search the codebase for the error path, read `_calculate_discount`, find the `TODO(SHOP-5102)` comment, and **edit the file to fix the bug**.
- **`query_database`** — your backend runs the SQL and returns rows. The agent never sees a connection string. We enforce `SELECT`-only on the server side, not just via prompting.
- **`search_past_incidents`** — your backend queries the issue tracker and returns matches. In production this would call Jira, Linear, or GitHub Issues.
- **`open_pull_request`** — the agent calls this after editing the code and producing a diff. Your backend creates the PR and returns a PR number. Auto-answered in the loop.
- **`create_triage_ticket`** — when the agent is ready to file, this tool pauses the session for human review. The ticket now includes the PR number. No ticket is created until you approve.

The system prompt is deliberately short: persona, workflow outline, and safety rules. The detailed investigation protocol lives in the Skill, keeping the prompt stable across agent versions.

In [2]:
SYSTEM_PROMPT = """\
You are a production bug investigator for the SmartCommerce platform.

When you receive a bug report, follow the investigation protocol in your
bug-investigator-runbook skill EXACTLY. Do not skip steps.

You have access to:
1. The codebase (mounted in the sandbox — use read, grep, bash, edit)
2. A read-only database query tool (query_database)
3. A past incident search tool (search_past_incidents)
4. A pull request tool (open_pull_request)
5. A ticket creation tool (create_triage_ticket) — requires human approval

IMPORTANT RULES:
- Only run SELECT-style read queries. Never attempt to modify data.
- Redact customer PII (emails, names) from your triage report.
- Always quantify the blast radius (how many records are affected).
- If uncertain about root cause, say so explicitly — do not guess.
- After identifying root cause, edit the file to fix the bug, save a backup
  first (cp file.py file.py.bak), then produce a unified diff (diff -u).
- Call open_pull_request with the diff before filing the ticket.
- When you have completed your investigation AND opened a PR, call
  create_triage_ticket with your full triage report including the PR number.
  This will pause for human approval.
"""

agent = client.beta.agents.create(
    name="cookbook-bug-investigator",
    model=MODEL,
    system=SYSTEM_PROMPT,
    skills=[
        {"type": "custom", "skill_id": skill.id, "version": skill.latest_version},
    ],
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
            ],
        },
        {
            "type": "custom",
            "name": "query_database",
            "description": (
                "Execute a read-only SQL query against the application database. "
                "Only SELECT statements are allowed. Returns columns and rows. "
                "Use this to investigate order state, audit logs, discount codes, "
                "and blast radius."
            ),
            "input_schema": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": "A SELECT query to run against the database.",
                    }
                },
                "required": ["sql"],
            },
        },
        {
            "type": "custom",
            "name": "search_past_incidents",
            "description": (
                "Search the issue tracker for past incidents similar to the "
                "current bug. Pass keywords related to the symptoms or root "
                "cause. Returns matching past incidents with their resolutions."
            ),
            "input_schema": {
                "type": "object",
                "properties": {
                    "keywords": {
                        "type": "string",
                        "description": "Space-separated keywords to search for.",
                    }
                },
                "required": ["keywords"],
            },
        },
        {
            "type": "custom",
            "name": "open_pull_request",
            "description": (
                "Open a pull request with the proposed code fix. Include the "
                "PR title, a body describing the root cause and fix, and the "
                "unified diff output. Returns a PR number."
            ),
            "input_schema": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "PR title (include bug ID).",
                    },
                    "body": {
                        "type": "string",
                        "description": "PR description: root cause summary and what the fix does.",
                    },
                    "diff": {
                        "type": "string",
                        "description": "Unified diff output from diff -u.",
                    },
                },
                "required": ["title", "body", "diff"],
            },
        },
        {
            "type": "custom",
            "name": "create_triage_ticket",
            "description": (
                "Create an enriched triage ticket with the investigation findings. "
                "This action requires human approval before execution. "
                "Include the full triage report with root cause, severity, "
                "blast radius, and recommended actions."
            ),
            "input_schema": {
                "type": "object",
                "properties": {
                    "bug_id": {"type": "string", "description": "The original bug report ID."},
                    "severity": {
                        "type": "string",
                        "enum": ["Critical", "High", "Medium", "Low"],
                        "description": "Classified severity level.",
                    },
                    "summary": {"type": "string", "description": "One-sentence root cause summary."},
                    "triage_report": {
                        "type": "string",
                        "description": "Full triage report in markdown format.",
                    },
                    "pr_number": {
                        "type": "integer",
                        "description": "PR number from open_pull_request.",
                    },
                    "recommended_assignee": {
                        "type": "string",
                        "description": "Team or person who should fix this.",
                    },
                },
                "required": ["bug_id", "severity", "summary", "triage_report", "pr_number"],
            },
        },
    ],
)
print(f"agent: {agent.id} v{agent.version}")

agent: agent_01JC8KA3FGH v1


## 3. Mount the codebase into the sandbox

The agent investigates code like a developer: `grep` for error messages, `read` the relevant files, `bash` to explore the directory tree. To make that work, we upload three fixture Python files and mount them at `src/smartcommerce/` inside a cloud environment.

These files are intentionally *excerpts* — they import modules (`database`, `notifications`, `audit_log`) that aren't present, because the agent only needs to read the discount logic, the data models, and the comments. In production you'd mount a full `github_repository` resource:

```python
{"type": "github_repository", "url": "https://github.com/your-org/smartcommerce",
 "authorization_token": os.environ["GITHUB_TOKEN"], "mount_path": "src"}
```

In [3]:
env = client.beta.environments.create(
    name="cookbook-bug-investigator-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)


def upload(path: Path, mime: str) -> str:
    with path.open("rb") as f:
        return client.beta.files.upload(file=(path.name, f, mime)).id


codebase_dir = FIXTURE / "fixtures" / "codebase"
file_ids: dict[str, str] = {}
for filepath in sorted(codebase_dir.glob("*.py")):
    file_ids[filepath.name] = upload(filepath, "text/x-python")

RESOURCES = [
    {"type": "file", "file_id": file_ids[name], "mount_path": f"src/smartcommerce/{name}"}
    for name in sorted(file_ids)
]
print(f"environment: {env.id}")
print(f"mounted {len(RESOURCES)} codebase files")

environment: env_01JC8KA8PQR
mounted 3 codebase files


## 4. Dispatch the bug report

In production the trigger would be a Jira webhook, a Zendesk automation, or a PagerDuty alert. The shape is always the same: create a session, mount the data, send the payload as `user.message`.

The `handle_bug_webhook` function below is the entire integration point. Everything else — code search, database queries, incident lookup, severity classification — the agent figures out by following the Skill.

> **Timing:** The first session on a new environment provisions a cloud sandbox (container spin-up, file mounts). Expect 2–5 minutes for the full investigation on a cold start. Subsequent sessions on the same environment are faster. The committed cell outputs below show what a completed run looks like.

In [4]:
def handle_bug_webhook(report: dict) -> str:
    """Create a session and send a bug report for investigation."""
    session = client.beta.sessions.create(
        environment_id=env.id,
        agent={"type": "agent", "id": agent.id, "version": agent.version},
        resources=RESOURCES,
        title=f"[{report['id']}] {report['summary'][:80]}",
    )
    client.beta.sessions.events.send(
        session.id,
        events=[
            {
                "type": "user.message",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "New production incident. Please investigate and triage:\n\n"
                            f"```json\n{json.dumps(report, indent=2)}\n```"
                        ),
                    }
                ],
            }
        ],
    )
    return session.id


session_id = handle_bug_webhook(BUG_REPORT)
print(f"session: {session_id}")

session: session_01JC8KAD4ST


## 5. The tool-call loop: auto-answer data + PR tools, pause on ticket creation

This is the core of the integration. The agent works in the sandbox autonomously — `read`, `grep`, `bash` calls happen inside the container and show up as `agent.tool_use` events (we just print them). When the agent needs information from *outside* the sandbox, it calls a custom tool and the session goes idle with `stop_reason.type == "requires_action"`.

The loop below handles four cases:

| Custom tool | What the loop does |
|-------------|--------------------|
| `query_database` | Validates the SQL is read-only, runs it against the mock DB, returns rows. |
| `search_past_incidents` | Keyword-matches against the fixture incident list, returns matches. |
| `open_pull_request` | Records the PR (title, body, diff) in a local list and returns a mock PR number. |
| `create_triage_ticket` | **Stops the loop** and returns the tool call ID so the next cell can approve or reject. |

The SQL guard (`_sql_looks_read_only`) is a heuristic — it allows `SELECT` and `WITH`, and rejects tokens like `INSERT`, `DROP`, `TRUNCATE`. In production, enforce this at the database layer (a read-only user or query proxy), not just in Python.

In [5]:
seen_events: set[str] = set()
custom_calls: dict[str, object] = {}
pending_triage: list[dict] = []
prs: list[dict] = []


def search_past_incidents_local(keywords: str) -> list:
    """Keyword search over the fixture incident list."""
    parts = [kw for kw in keywords.lower().split() if kw]
    if not parts:
        return []
    out = []
    for incident in PAST_INCIDENTS:
        hay = f"{incident['summary']} {incident['root_cause']} {incident['resolution']}".lower()
        if any(kw in hay for kw in parts):
            out.append(incident)
    return out


def _sql_looks_read_only(sql: str) -> bool:
    """Heuristic guard: allow SELECT / WITH … SELECT; reject write statements."""
    stripped = sql.strip()
    if not stripped:
        return False
    upper = stripped.upper()
    if not (upper.startswith("SELECT") or upper.startswith("WITH")):
        return False
    normalized = f" {upper} "
    banned = (
        " INSERT ", " UPDATE ", " DELETE ", " DROP ", " TRUNCATE ", " ALTER ",
        " CREATE ", " GRANT ", " REVOKE ", " EXEC ", " EXECUTE ", " MERGE ",
        " CALL ", " PRAGMA ",
    )
    return not any(token in normalized for token in banned)


def handle_custom_tool(name: str, args: dict):
    """Route a custom tool call to the appropriate mock handler."""
    if name == "query_database":
        sql = args.get("sql", "")
        if not _sql_looks_read_only(sql):
            return {"error": "Only read-only SELECT-style queries are allowed."}
        return execute_mock_query(sql)
    if name == "search_past_incidents":
        return search_past_incidents_local(args.get("keywords", ""))
    if name == "open_pull_request":
        n = len(prs) + 1
        url = f"mock://smartcommerce/pull/{n}"
        prs.append({"number": n, "url": url, **args})
        print(f"\n\u2500\u2500 PR #{n}: {args['title']} \u2500\u2500")
        return {"pr_number": n, "url": url}
    if name == "create_triage_ticket":
        return {"status": "pending_approval"}
    return {"error": f"Unknown tool: {name}"}


def run_investigation(session_id: str) -> str | None:
    """Drive the session until the agent files a ticket (returns tool ID) or ends its turn."""
    responded: set[str] = set()
    while True:
        idle_stop = None
        for ev in client.beta.sessions.events.list(session_id):
            if ev.id in seen_events:
                continue
            seen_events.add(ev.id)
            if ev.type == "agent.message":
                for block in ev.content:
                    if block.type == "text":
                        print(block.text, end="")
            elif ev.type == "agent.tool_use":
                print(f"\n  [{ev.name}]")
            elif ev.type == "agent.custom_tool_use":
                custom_calls[ev.id] = ev
                print(f"\n\u2192 {ev.name}")
            elif ev.type == "session.status_idle":
                idle_stop = ev.stop_reason
            elif ev.type == "session.status_terminated":
                return None
        if idle_stop is None:
            time.sleep(1.0)
            continue
        if idle_stop.type == "end_turn":
            return None
        if idle_stop.type == "requires_action":
            for event_id in idle_stop.event_ids:
                if event_id in responded:
                    continue
                call = custom_calls[event_id]
                if call.name == "create_triage_ticket":
                    pending_triage.append({"tool_use_id": event_id, **call.input})
                    return event_id
                payload = handle_custom_tool(call.name, call.input)
                client.beta.sessions.events.send(
                    session_id,
                    events=[
                        {
                            "type": "user.custom_tool_result",
                            "custom_tool_use_id": event_id,
                            "content": [{"type": "text", "text": json.dumps(payload)}],
                        }
                    ],
                )
                responded.add(event_id)
        time.sleep(1.0)


approval_tool_id = run_investigation(session_id)

I'll investigate this bug report systematically following the investigation protocol.

## Step 1: Parse the Bug Report

- Error symptom: Checkout crashes with "Payment Error: Invalid amount"
- Affected entity: Order ORD-20260408-7721, Customer CUST-88291
- Environment: Production (us-east-1)
- Timestamp: 2026-04-08T14:23:00Z
- Impact scope: 4 customers affected in the last hour

## Step 2: Search the Codebase

  [grep]
  [read]
  [read]
  [read]

Found the discount calculation in `order_service.py`. The `_calculate_discount` method
for `tiered_percent` type applies per-item tier percentages without capping. There's a
`TODO(SHOP-5102)` in `discount_engine.py` and a `BUG:` comment in `order_service.py`
confirming this is a known issue.

## Step 3: Query the Database

→ query_database
→ query_database
→ query_database
→ query_database
→ query_database
→ query_database

Key findings:
- Order subtotal: $52.45, discount applied: $78.68, total: -$26.23
- BULK25 is tiered_percent with tiers [[

## 6. Review the triage report and proposed PR

The agent has finished its investigation and opened a PR. Before anything gets filed, you see exactly what it found — severity, root cause, blast radius, the proposed PR, and the full markdown report. In production this is the payload you'd drop into Slack with **Approve** and **Reject** buttons, or surface in a review dashboard.

In [6]:
if approval_tool_id and pending_triage:
    t = pending_triage[0]
    print(f"Bug ID:   {t.get('bug_id', 'N/A')}")
    print(f"Severity: {t.get('severity', 'N/A')}")
    print(f"Summary:  {t.get('summary', 'N/A')}")
    print(f"PR:       #{t.get('pr_number', 'N/A')}")
    print(f"Assignee: {t.get('recommended_assignee', 'N/A')}")
    if prs:
        print(f"PR Title: {prs[-1]['title']}")
        print(f"PR URL:   {prs[-1].get('url', 'N/A')}")
    print("\n" + "─" * 60 + "\n")
    print(t.get("triage_report", "(no report)"))

Bug ID:   SHOP-5102
Severity: Critical
Summary:  tiered_percent discount in _calculate_discount applies per-item percentage without capping at MAX_DISCOUNT_PERCENT, producing negative totals for orders with many low-priced items
PR:       #1
Assignee: payments-team
PR Title: Fix SHOP-5102: Cap tiered_percent discount at MAX_DISCOUNT_PERCENT
PR URL:   mock://smartcommerce/pull/1

────────────────────────────────────────────────────────────

## Triage Report: SHOP-5102

### Summary
The `tiered_percent` discount type in `_calculate_discount` applies the tier
percentage to each item’s (price × quantity) without capping the total discount
at `MAX_DISCOUNT_PERCENT` (50%) of the subtotal.

### Severity: Critical
Negative monetary amounts in financial transactions affecting 12 production
orders. No workaround — all BULK25 orders with 15+ cheap items are affected.

### Root Cause Analysis
- `order_service.py`, `_calculate_discount`, lines 154–165
- For `tiered_percent`, the code iterates items 

## 7. Approve (or reject) the ticket and PR

Send your decision back as the `create_triage_ticket` tool result. The JSON payload can include severity overrides or notes — the agent sees them and can adjust its final confirmation message.

To try a rejection, change `"status"` to `"rejected"` and add a `"notes"` field explaining what needs more investigation.

In [7]:
if approval_tool_id:
    decision = {
        "status": "approved",
        "severity": pending_triage[0].get("severity"),
        "notes": "Confirmed. Root cause matches the code. Assign to payments team.",
    }
    client.beta.sessions.events.send(
        session_id,
        events=[
            {
                "type": "user.custom_tool_result",
                "custom_tool_use_id": approval_tool_id,
                "content": [{"type": "text", "text": json.dumps(decision)}],
            }
        ],
    )
    # Let the agent post its confirmation.
    run_investigation(session_id)

Ticket SHOP-5102 has been approved and filed as Critical.
PR #1 is ready for code review by the payments team.


## 8. Generate a data remediation script

The code fix (PR) prevents new orders from hitting the bug, but the 12 orders already corrupted still have negative totals in the database. Instead of writing the remediation SQL yourself, ask the agent — it already knows the root cause, the affected table, the blast-radius query, and the correct discount-cap logic.

We send a follow-up message on the same session. The agent still has full context from the investigation: the code it read, the queries it ran, the diff it produced. It writes the remediation script in the sandbox and prints it.

In [8]:
if approval_tool_id:
    client.beta.sessions.events.send(
        session_id,
        events=[
            {
                "type": "user.message",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Ticket approved. Now generate a production data remediation "
                            "SQL script to fix the existing affected orders.\n\n"
                            "Requirements:\n"
                            "1. Find all orders where discount_code = 'BULK25' AND total < 0\n"
                            "2. Recalculate the discount amount using the corrected logic "
                            "(cap at MAX_DISCOUNT_PERCENT = 50% of subtotal)\n"
                            "3. Update the total = subtotal - capped_discount\n"
                            "4. Log each correction in the audit_log\n"
                            "5. Wrap everything in a transaction with a dry-run mode\n\n"
                            "Write the script to remediation.sql in the sandbox, then "
                            "print the full contents so I can review it."
                        ),
                    }
                ],
            }
        ],
    )
    run_investigation(session_id)
    print("\n\n" + "─" * 60)
    print("Remediation script generated. Review before running in production.")

  [edit]
  [bash]

I’ve written the remediation script to `remediation.sql`. Here are the contents:

```sql
-- Remediation script for SHOP-5102
-- Generated: 2026-04-12
-- Affected rows: 12
-- Dry-run mode: set @dry_run = 1 to preview without committing

SET @dry_run = 0;

BEGIN TRANSACTION;

-- 1. Preview affected records
SELECT order_id, subtotal, discount_amount, total,
       LEAST(discount_amount, subtotal * 0.50) AS corrected_discount,
       subtotal - LEAST(discount_amount, subtotal * 0.50) AS corrected_total
FROM orders
WHERE discount_code = 'BULK25' AND total < 0;

-- 2. Apply corrections
UPDATE orders
SET discount_amount = LEAST(discount_amount, subtotal * 0.50),
    total = subtotal - LEAST(discount_amount, subtotal * 0.50),
    status = 'pending'
WHERE discount_code = 'BULK25' AND total < 0;

-- 3. Audit log
INSERT INTO audit_log (event_type, timestamp, entity_id, details)
SELECT 'data_remediation.SHOP-5102', NOW(), order_id,
       JSON_OBJECT('old_total', total, 'new_tot

## 9. Audit trail and cleanup

Every `read`, `grep`, `bash`, `edit`, database query, PR creation, incident search, and approval decision is persisted as a session event — open the [Console](https://platform.claude.com/) under **Managed Agents → Sessions** for the complete trace. No custom logging code needed.

In [9]:
wait_for_idle_status(client, session_id)
client.beta.sessions.archive(session_id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
client.beta.skills.versions.delete(skill.latest_version, skill_id=skill.id)
client.beta.skills.delete(skill.id)
print("archived")

archived


## What you learned

In this cookbook you:

1. **Uploaded a Skill** that encodes your team’s investigation protocol, severity matrix, SQL templates, fix guidelines, and remediation script template — portable across Managed Agents, the Agent SDK, and Claude Code.
2. **Created an agent** that combines sandbox tools (`read`, `grep`, `bash`, `edit`) with four custom tools (`query_database`, `search_past_incidents`, `open_pull_request`, `create_triage_ticket`).
3. **Built a tool-call loop** that auto-answers data and PR tools, enforces read-only SQL on your server, and pauses at `create_triage_ticket` for human review.
4. **Ran an end-to-end investigation** where the agent found the root cause, quantified the blast radius (12 orders), cross-referenced a past incident (SHOP-4891), proposed a code fix, opened a PR, and filed a triage report — all from a single bug report.
5. **Generated a data remediation script** to fix the corrupted records, with dry-run mode and audit logging.
6. **Reviewed the full audit trail** in the Console — every tool call, every decision, every approval — without writing custom logging code.

## Going to production

This notebook mocks three external services. Here's how to swap each one in.

### Database

Replace `execute_mock_query` with a call to your read-only query service. Keep `_sql_looks_read_only` (or a real SQL parser) on *your* server — defense in depth, not just prompting. If your database uses integrated auth (e.g. Windows Trusted Connection with SQL Server), consider running the agent via the [Agent SDK](https://docs.claude.com/claude/docs/agent-sdk) on your infrastructure, where the process inherits the triggering user's identity and the DB enforces their permissions directly.

### Issue tracker

Point `search_past_incidents` at Jira's search API, Linear's GraphQL endpoint, or GitHub Issues. Return the same shape (id, summary, status, root_cause, resolution) so the Skill's cross-referencing step works unchanged.

### Ticket creation & approval

When `create_triage_ticket` pauses the session, post the report to Slack or Teams with **Approve / Reject** buttons and send the `user.custom_tool_result` from your button handler. The [`slack_data_bot` cookbook](slack_data_bot.ipynb) shows the Bolt wiring for this pattern. Alternatively, register a Console webhook on `session.status_idled` — see [`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb) for the webhook setup — to avoid holding an HTTP connection while humans think.

### Pull requests

The mock `open_pull_request` tool writes to a local list. In production, replace it with a call to your Git hosting API (GitHub, GitLab, Bitbucket). Alternatively, drop the custom tool entirely and give the agent the GitHub MCP server instead — [`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb) walks through per-user credentials:

```python
agent = client.beta.agents.create(
    ...,
    mcp_servers=[{"type": "url", "name": "github",
                  "url": "https://mcp.github.com/sse",
                  "authorization_token": os.environ["GITHUB_TOKEN"]}],
)
```

When the agent has native MCP access to GitHub, it can open real PRs directly from the sandbox diff. The `create_triage_ticket` human gate still blocks — so no ticket is filed (and the team isn’t pinged to merge) until a human says yes.

### Keeping the Skill portable

The same `SKILL.md` works in three contexts without modification:

- **Managed Agents** — uploaded via the Skills API (this notebook).
- **Agent SDK** — referenced as a local file path when running on your own infrastructure.
- **Claude Code** — developers use it interactively during local debugging.

Author the runbook once; deploy it everywhere.

### Data remediation

The agent produces a SQL remediation script in the sandbox. In production, pipe this through your change-management workflow: peer review the SQL, run it against a staging replica first, then execute in production inside a transaction. The agent’s script includes a dry-run mode (`@dry_run = 1`) so the DBA can verify row counts before committing.

For teams that use a migration framework (Flyway, Alembic, Liquibase), have the agent write the output as a versioned migration file instead of raw SQL. Update the Skill’s Step 9 template to match your migration format.

## Evaluating before you ship

Before deploying, validate against your resolved bug backlog:

1. Pick 10–20 closed bugs where the root cause is known.
2. Feed the agent **only** the original report (no resolution notes).
3. Compare its triage report against the actual resolution. Score: root cause found? Severity correct? Right team routing?
4. Track accuracy over time as you refine the Skill — common improvements are adding query templates for patterns the agent missed, severity examples for edge cases, and a "known pitfalls" section for recurring false positives.